# 🎯 Hyperparameter Optimization using Optuna

## Enterprise AutoML Platform

## 📖 About this Notebook

This notebook demonstrates automatic hyperparameter optimization using Optuna.

The optimization process searches for the best parameter combinations to maximize model performance while reducing manual tuning efforts.

Sections

1. Title
2. Objective
3. Import Libraries
4. Load Dataset
5. Data Preprocessing
6. Train-Test Split
7. Build XGBoost Model
8. Define Optuna Objective
9. Run Optimization
10. Best Parameters
11. Train Best Model
12. Evaluate Model
13. Feature Importance
14. Optimization History
15. Conclusion

### Objective

This notebook demonstrates automatic hyperparameter optimization using **Optuna**.

The optimization process helps improve model performance by automatically searching for the best hyperparameter combinations.

Model Used:
- XGBoost Classifier

Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import optuna
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

import optuna.visualization as vis

Load Dataset

In [ ]:
df = pd.read_csv(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

Encode Target

In [ ]:
df["Churn"] = df["Churn"].map(
    {
        "No":0,
        "Yes":1,
    }
)

Features

In [ ]:
X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

Numerical Columns

In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

Categorical Columns

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

Train Test Split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        ),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                ]
            ),
            numerical_columns,
        ),
        (
            "cat",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ]
)

X_train = preprocessor.fit_transform(
    X_train
)

X_test = preprocessor.transform(
    X_test
)

Objective Function

In [ ]:
def objective(trial):

    params = {

        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            50,
            100,
        ),

        "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            6,
        ),

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
        ),

        "subsample":
        trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),

        "colsample_bytree":
        trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),

        "eval_metric":"logloss",

        "verbosity":0,

        "random_state":42,
    }

    model = XGBClassifier(
        **params
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
    ).mean()

    return score

Run Optimization

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=20,
)

Best Score

In [ ]:
study.best_value

Best Parameters

In [ ]:
study.best_params

Train Best Model

In [ ]:
best_model = XGBClassifier(
    **study.best_params,
    eval_metric="logloss",
    verbosity=0,
    random_state=42,
)

best_model.fit(
    X_train,
    y_train,
)

Prediction

In [ ]:
prediction = best_model.predict(
    X_test
)

accuracy_score(
    y_test,
    prediction,
)

Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = pd.DataFrame(
    {
        "Feature":
        preprocessor.get_feature_names_out(),

        "Importance":
        best_model.feature_importances_,
    }
)

importance = importance.sort_values(
    by="Importance",
    ascending=False,
)

importance.head(20)

Plot

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    importance.head(20)["Feature"],
    importance.head(20)["Importance"],
)

plt.gca().invert_yaxis()

plt.title(
    "Top 20 Important Features"
)

plt.show()

Optuna History

In [ ]:
vis.plot_optimization_history(
    study
)

Parameter Importance

vis.plot_param_importances(
    study
)

# ✅ Conclusion

Completed

- Hyperparameter Optimization
- Automatic Parameter Search
- Cross Validation
- Best Parameter Selection
- Model Retraining
- Feature Importance Analysis
- Optimization History Visualization

The optimized model can now be used in the production AutoML pipeline.

---

# 🏭 Production Implementation

The Enterprise AutoML Platform integrates Optuna directly into the training workflow.

Production capabilities include:

- Automatic Hyperparameter Optimization
- Cross Validation
- Best Parameter Selection
- Optimized Model Training

These steps execute automatically during production training.